In [1]:
# import dependencies
import pandas as pd
import re
import numpy as np
from rapidfuzz import process

In [2]:
# load acs data (for merging town names, GEOID, population data)
acs_df = pd.read_csv("../data/cleaned/acs_summary.csv")

Used this [list of HMGP activities](https://www.fema.gov/sites/default/files/2020-08/fema_mt-egrants-guide-to-eligible-activities-and-codes_job_aid_March_2018.pdf) to create a csv to classify projectType.

In [3]:
# load FEMA project codes (for categorizing projects)
codes_df = pd.read_csv("../data/resources/FEMA_project_codes.csv")
codes_df.head()

,code,main_category,subcategory,flood_mitigation_flag
0,100.1,planning,public_awareness,0
1,103.1,planning,feasibility_study,0
2,104.1,planning,codes_standards,0
3,106.1,planning,other_nonconstruction,0
4,106.2,planning,other_nonconstruction,0


In [4]:
# load FEMA Hazard Mitigation Assistance Projects for Vermont (the data of interest)
df = pd.read_csv("../data/raw/funding/vt_hma_awards.csv")
print(df.shape)
df.head()

(756, 33)


,projectIdentifier,programArea,programFy,region,state,stateNumberCode,county,countyCode,disasterNumber,projectCounties,...,federalShareObligated,subrecipientAdminCostAmt,srmcObligatedAmt,recipientAdminCostAmt,costSharePercentage,benefitCostRatio,netValueBenefits,numberOfFinalProperties,numberOfProperties,id
0,DR-4532-0005-R,HMGP,2020,1,Vermont,50,Rutland,21.0,4532.0,RUTLAND,...,140188.70,0.0,0.0,0.0,0.90,1.13,323000.0,1,1,344986e8-9849-4f9a-9504-436d4a3c92c7
1,DR-4720-0047-R,HMGP,2023,1,Vermont,50,Washington,23.0,4720.0,WASHINGTON,...,237540.75,0.0,0.0,0.0,0.75,0.00,0.0,1,1,1bce9785-0760-4558-bf79-6cd8047bad9b
2,DR-4720-0069-R,HMGP,2023,1,Vermont,50,Washington,23.0,4720.0,WASHINGTON,...,265767.75,0.0,0.0,0.0,0.75,0.00,0.0,1,1,62892290-55db-4565-90e1-29436e27138c
3,DR-4720-0043-R,HMGP,2023,1,Vermont,50,Washington,23.0,4720.0,WASHINGTON,...,121983.00,0.0,0.0,0.0,0.75,0.00,0.0,1,1,65ee2247-4a87-42b0-a1d0-70c290c0a844
4,DR-4022-0109-R,HMGP,2011,1,Vermont,50,Washington,23.0,4022.0,WASHINGTON,...,0.00,0.0,0.0,0.0,0.75,0.00,0.0,0,1,d3018dab-e76a-4236-8d76-a5da6fc581ff


In [5]:
# prune columns to those needed for analysis
df = df[
    [
        "projectIdentifier",
        "programArea",
        "programFy",
        "disasterNumber",
        "projectCounties",
        "projectType",
        "status",
        "subrecipient",
        "dateApproved",
        "projectAmount",
        "federalShareObligated",
        "costSharePercentage",
        "benefitCostRatio",
        "netValueBenefits",
        "numberOfFinalProperties",
        "numberOfProperties",
    ]
]
df.shape

(756, 16)

In [6]:
# merge project codes csv with FEMA HMA dataframe, including handling multiple codes in projectType

# extract all codes from projectType, creating a list of codes for each project
# regex: one or more digits, followed by a period, followed by one or more digits (e.g. 200.1)
df = df.copy()  # avoid SettingWithCopyWarning
df["codes"] = df["projectType"].str.findall(r"\d+\.\d+")

# explode codes so each project-code is a row
df_exploded = df.explode("codes")

# ensure string type for merging
df_exploded["codes"] = df_exploded["codes"].astype(str)
codes_df["code"] = codes_df["code"].astype(str)

# merge exploded df with project type codebook
df_merged = df_exploded.merge(codes_df, left_on="codes", right_on="code", how="left")

# create dummies for main_category and concatenate back to df
category_dummies = pd.get_dummies(df_merged["main_category"])
df_merged = pd.concat([df_merged, category_dummies], axis=1)

# create {main_category}_funding columns for each main_category
df_merged["n_codes"] = df_merged.groupby("projectIdentifier")["codes"].transform(
    "count"
)
# split funding equally across codes for each project, then multiply by category dummies to get funding for each main category
df_merged["allocated_funding"] = (
    df_merged["federalShareObligated"] / df_merged["n_codes"]
)
for col in category_dummies.columns:
    df_merged["funding_" + col] = df_merged[col] * df_merged["allocated_funding"]

# aggregate back to project level,
# with flags for each main category and flood mitigation, and concatenated subcategories
agg = (
    df_merged.groupby("projectIdentifier")
    .agg(
        {
            # for projects with multiple codes, flag if any code has a flood mitigation flag of 1
            "flood_mitigation_flag": "max",
            # join all unique, non-null subcategories for the project
            "subcategory": lambda x: ",".join(sorted(set(x.dropna()))),
            # old approach, superseded by dummies, left for testing
            # "main_category": lambda x: ",".join(sorted(set(x.dropna()))),
            # flag if any code has a 1 for each main category
            **{col: "max" for col in category_dummies.columns},
            # sum the funding for each main category across all codes for the project
            **{"funding_" + col: "sum" for col in category_dummies.columns},
        }
    )
    .reset_index()
)

# merge back to original df
df_coded = df.merge(agg, on="projectIdentifier", how="left")
df = df_coded
print(df.shape)
df.head()

(756, 39)


,projectIdentifier,programArea,programFy,disasterNumber,projectCounties,projectType,status,subrecipient,dateApproved,projectAmount,...,funding_acquisition_buyouts,funding_admin,funding_ecosystem_restoration,funding_equipment,funding_flood_control,funding_infrastructure_protection,funding_infrastructure_stormwater,funding_other,funding_planning,funding_structure_protection
0,DR-4532-0005-R,HMGP,2020,4532.0,RUTLAND,200.1: Acquisition of Private Real Property (S...,Closed,Pittsfield (Town of),2025-01-13T00:00:00.000Z,155765.23,...,140188.70,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,DR-4720-0047-R,HMGP,2023,4720.0,WASHINGTON,200.1: Acquisition of Private Real Property (S...,Approved,Statewide,2024-11-14T00:00:00.000Z,316721.00,...,237540.75,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,DR-4720-0069-R,HMGP,2023,4720.0,WASHINGTON,200.1: Acquisition of Private Real Property (S...,Approved,Statewide,2025-04-21T00:00:00.000Z,354357.00,...,265767.75,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,DR-4720-0043-R,HMGP,2023,4720.0,WASHINGTON,200.1: Acquisition of Private Real Property (S...,Approved,Statewide,2024-11-14T00:00:00.000Z,162644.00,...,121983.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,DR-4022-0109-R,HMGP,2011,4022.0,WASHINGTON,202.1: Elevation of Private Structures - Riverine,Closed,Waterbury (Town of),2016-05-11T00:00:00.000Z,0.00,...,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [7]:
df["flood_mitigation_flag"].value_counts()

flood_mitigation_flag
1.0    509
0.0    234
Name: count, dtype: int64

In [8]:
# create cost per property columns, may be useful? say, for property acquisition projects?
df["cost_per_property"] = np.where(
    df["numberOfProperties"] > 0,
    df["federalShareObligated"] / df["numberOfProperties"],
    np.nan,
)
df["cost_per_final_property"] = np.where(
    df["numberOfFinalProperties"] > 0,
    df["federalShareObligated"] / df["numberOfFinalProperties"],
    np.nan,
)
df.shape

(756, 41)

In [9]:
# dict for manual town assignments based on subrecipient names
manual_town_dict = {
    # villages inside of towns
    "JEFFERSONVILLE": "CAMBRIDGE TOWN",
    "LYNDONVILLE": "LYNDON TOWN",
    "ENOSBURG FALLS": "ENOSBURGH TOWN",
    # easily identifiable with a certain town
    "RUPERT- HIGHWAY DEPARTMENT": "RUPERT TOWN",
    "STOWE ELECTRIC DEPARTMENT": "STOWE TOWN",
    "MONTPELIER- PUBLIC WORKS DEPARTMENT": "MONTPELIER CITY",
    "BRATTLEBORO HOUSING AUTHORITY": "BRATTLEBORO TOWN",
    "BARRE HISTORICAL SOCIETY INC": "BARRE CITY",
    "ETHAN ALLEN HOMESTEAD TRUST": "BURLINGTON CITY",
    "NORWICH UNIVERSITY": "NORTHFIELD TOWN",
    # changed it's name in 1999, and the solitary project for this subrecipient is from 1996
    "SHERBURNE": "KILLINGTON TOWN",
}


# Barre, Rutland, St. Albans, and Newport have a city and town of the same name
# Vermont is sometimes lacking in creativity
# override function for classifying a few towns with ambiguous names, based on explicit mentions of "TOWN" or "CITY" in the subrecipient name
def classify_barre_rutland_stalbans(subrecipient):
    if pd.isna(subrecipient):
        return None
    s = subrecipient.upper().replace(".", "")
    ambiguous_towns = {
        "BARRE": ["BARRE TOWN", "BARRE CITY", "AMBIGUOUS: BARRE"],
        "RUTLAND": ["RUTLAND TOWN", "RUTLAND CITY", "AMBIGUOUS: RUTLAND"],
        "ST ALBANS": ["ST. ALBANS TOWN", "ST. ALBANS CITY", "AMBIGUOUS: ST. ALBANS"],
        "NEWPORT": ["NEWPORT TOWN", "NEWPORT CITY", "AMBIGUOUS: NEWPORT"],
    }
    for town, [town_name, city_name, ambiguous_name] in ambiguous_towns.items():
        if town in s:
            if "TOWN" in s:
                return town_name
            elif "CITY" in s:
                return city_name
            elif s.strip() == town:
                return ambiguous_name
    return None

In [10]:
# regex to clean subrecipient names
def clean_town(x):
    # handle NaN values
    if pd.isna(x):
        return x
    # convert to uppercase
    x = x.upper()
    # remove common prefixes and suffixes
    x = re.sub(r"\(TOWN OF\)|\(CITY OF\)|\(VILLAGE OF\)", "", x)
    x = re.sub(r"TOWN OF|CITY OF|VILLAGE OF", "", x)
    # remove any remaining parentheses
    x = re.sub(r"\(", "", x)
    x = re.sub(r"\)", "", x)
    # remove punctuation, extra whitespace, leading/trailing whitespace
    x = re.sub(r",", "", x)
    x = re.sub(r"\s+", " ", x)
    x = x.strip()
    # remove ' VT' or ' VERMONT' at the end, unless preceded by ' OF '
    # cleans some towns w/o affecting "Univ of Vermont", "State of Vermont", etc.
    # (?<! OF) means "not immediately preceded by ' OF'"
    x = re.sub(r"(?<! OF) (VT|VERMONT)$", "", x)
    return x


# apply cleaning function to subrecipient column to create town_clean
df["town_clean"] = df["subrecipient"].apply(clean_town)

# get and prepare canonical town list from ACS
town_list = acs_df["town_name"].str.upper().str.strip().unique().tolist()
# build dict of base names for structured fuzzy matching of town names that differ only by "TOWN", "CITY", or "VILLAGE"
town_list_base = {}
for t in town_list:
    parts = t.split()
    if len(parts) > 1:
        base = re.sub(r" (TOWN|CITY|VILLAGE)$", "", t)
        town_list_base[base] = t


# function for matching
# checking for exact matches, base name matches, and then fuzzy matches
# uses rapidfuzz for fuzzy matches, with a score cutoff to avoid bad matches
def match_town(name, town_list, score_cutoff=90):
    # handle NaN values
    if pd.isna(name):
        return None
    # check for exact match
    if name in town_list:
        return name
    # check for base match if name ends with " TOWN", " CITY", or " VILLAGE"
    if name in town_list_base:
        return town_list_base[name]
    # if no exact or base match, use fuzzy matching with a score cutoff to avoid bad matches
    # the fuzzy match, in this case, is redundant - exact and base matches catch all cases for Vermont
    match = process.extractOne(name, town_list, score_cutoff=score_cutoff)
    if match:
        return match[0]
    return None


# apply fuzzy matching to create town_match column
df["town_match"] = df["town_clean"].apply(lambda x: match_town(x, town_list))


# handle ambiguous cases, otherwise use fuzzy match
def resolve_town(row):
    # check for explicit classifications of ambiguous cases
    special = classify_barre_rutland_stalbans(row["subrecipient"])
    if special is not None:
        return special
    # then check manual overrides
    cleaned = row["town_clean"]
    if cleaned in manual_town_dict:
        return manual_town_dict[cleaned]
    # then check fuzzy match results
    elif pd.notna(row["town_match"]):
        return row["town_match"]
    else:
        return None


# create final town column using ambiguous case resolution and fuzzy matching
df["town_assigned"] = df.apply(resolve_town, axis=1)
print(df.shape)
df["town_assigned"].value_counts(dropna=False)

(756, 44)


town_assigned
None               271
PAWLET TOWN         16
MONTPELIER CITY     14
RUPERT TOWN         13
NORTHFIELD TOWN     12
                  ... 
WOODFORD TOWN        1
SHREWSBURY TOWN      1
BARRE CITY           1
DOVER TOWN           1
GUILDHALL TOWN       1
Name: count, Length: 149, dtype: int64

Re: unmatched

Need to resolve ambiguity for Barre / Rutland / St Albans - mostly property acquisition (planning for St Albans) - could merge city and town? but that merges acs variables?  
Orleans could be a village in Barton (a generator) or the county (a buyout)  
Statewide stuff? Might just drop it. Could distribute per-capita.  
Chittenden, Lamoille, Windham, Windsor, Ottauquechee, could be distributed across the respective county / multiple towns. Could be dropped.  

In [11]:
# review ambiguous and unmatched cases

# # review ambiguous cases (Rutland, Barre, St. Albans, and Newport, which have a city and town of the same name)
# ambig = df[df["town_assigned"].str.startswith("AMBIGUOUS", na=False)]
# print("Ambiguous subrecipients:", ambig[["subrecipient", "town_clean"]])

# # review unmatched cases
# pd.set_option("display.max_rows", None)
# # display where town_assigned is null to check unmatched cases
# unmatched = df[df["town_assigned"].isna()]
# print(
#     f'\n\nNumber of unmatched cases: {len(unmatched[["subrecipient", "town_clean", "town_match", "town_assigned"]].drop_duplicates().sort_values("town_clean"))}'
# )
# unmatched[
#     ["subrecipient", "town_clean", "town_match", "town_assigned"]
# ].drop_duplicates().sort_values("town_clean")

In [12]:
# pd.reset_option("display.max_rows")

In [13]:
# drop columns used for matching but not needed for analysis
df = df.drop(
    [
        "town_clean",
        "town_match",
        "codes",
    ],
    axis=1,
)
df.shape

(756, 41)

In [14]:
# # boolean based on whether we were able to assign a town or not
# df["is_town_assigned"] = df["town_assigned"].notna()
# df["is_town_assigned"].value_counts()

In [15]:
# assign a geographic level to each project, based on if a town was assigned and the subrecipient name
def classify_geo_level(row):
    if pd.notna(row.get("town_assigned")):
        return "local"
    s = str(row.get("subrecipient", "")).upper()
    # known county/regional entities in Vermont, based on manual review of subrecipient names
    if "COUNTY" in s or "REGION" in s or "REGIONAL" in s or "COMMISSION" in s:
        return "regional"
    # based on manual review of subrecipient names
    # exclude university from statewide
    if (
        (
            "STATE" in s
            or "VERMONT" in s
            or "VT" in s
            or "PUBLIC" in s
            or "ENVIRONMENTAL" in s
        )
        and "UNIV" not in s
        and "UNIVERSITY" not in s
    ):
        return "statewide"
    return "unknown"


df["geo_level"] = df.apply(classify_geo_level, axis=1)
df.geo_level.value_counts()

geo_level
local        485
statewide    220
unknown       33
regional      18
Name: count, dtype: int64

In [ ]:
# export df to csv
df.to_csv("../data/cleaned/fema_hma_projects_clean.csv", index=False)